In [2]:
import os
import sys
from pyspark.sql import SparkSession

In [3]:
# 1. 커널 재시작 후 가장 먼저 실행하세요!
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [4]:
# 기존 세션이 있다면 종료
try:
    spark.stop()
except:
    pass

In [5]:
os.environ["JAVA_HOME"] = "C:/tools/jdk17" # 설치하신 17 경로
os.environ["HADOOP_HOME"] = "C:/tools/hadoop-3.2.0"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "/bin;" + os.environ["PATH"]

In [47]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.python.worker.reuse", "true") \
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true") \
    .getOrCreate()

print("성공! 이제 Spark를 사용할 수 있습니다.")

성공! 이제 Spark를 사용할 수 있습니다.


In [7]:
import requests
import gzip
import shutil

In [26]:
# 1. CSV 압축 파일 다운로드
url = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz"
file_gz = "fhvhv_tripdata_2021-01.csv.gz"
file_csv = "fhvhv_tripdata_2021-01.csv"

In [27]:
print("다운로드 시작... (시간이 조금 걸릴 수 있습니다)")
response = requests.get(url, stream=True)
with open(file_gz, 'wb') as f:
    f.write(response.content)

다운로드 시작... (시간이 조금 걸릴 수 있습니다)


In [28]:
print("압축 푸는 중...")
with gzip.open(file_gz, 'rb') as f_in:
    with open(file_csv, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

print(f"완료! {file_csv} 파일이 준비되었습니다.")

압축 푸는 중...
완료! fhvhv_tripdata_2021-01.csv 파일이 준비되었습니다.


In [29]:
!head -n 101 fhvhv_tripdata_2021-01.csv > head.csv

In [24]:
import pandas as pd

In [25]:
df_pandas = pd.read_csv('head.csv')

In [26]:
df_pandas.dtypes

hvfhs_license_num        object
dispatching_base_num     object
pickup_datetime          object
dropoff_datetime         object
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [27]:
# Pandas를 거치지 않고 Spark가 직접 CSV를 읽습니다. 
# 이 방식은 파이썬 버전 호환성 문제에서 훨씬 자유롭습니다.
df_spark = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv('head.csv')

In [28]:
# 확인
df_spark.show()
df_spark.printSchema()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [29]:
df_spark.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', StringType(), True)])

In [8]:
from pyspark.sql import types

In [9]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True), 
    types.StructField('pickup_datetime', types.TimestampType(), True), 
    types.StructField('dropoff_datetime', types.TimestampType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('SR_Flag', types.StringType(), True)
])

In [10]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [11]:
df.head(10)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DOLocationID=167, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 23, 56), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 38, 5), PULocationID=233, DOLocationID=142, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 42, 51), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 45, 50), PULocationID=142, DOLocationID=143, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_dat

In [34]:
df = df.repartition(24)

In [35]:
import shutil
import os
if os.path.exists('fhvhv/2021/01/'):
    shutil.rmtree('fhvhv/2021/01/')

In [36]:
df.write.parquet('fhvhv/2021/01/')

In [12]:
df = spark.read.parquet('fhvhv/2021/01/')

In [18]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [ ]:
SELECT * from df WHERE hvfhs_license_num = HV0003

In [33]:
from pyspark.sql import functions as F

In [38]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02875|2021-01-06 10:38:29|2021-01-06 10:45:44|         259|         259|   NULL|
|           HV0003|              B02887|2021-01-05 20:42:51|2021-01-05 20:58:14|          75|         147|   NULL|
|           HV0003|              B02887|2021-01-04 16:21:23|2021-01-04 16:29:28|          61|          72|   NULL|
|           HV0005|              B02510|2021-01-05 13:18:03|2021-01-05 13:40:26|           7|         137|   NULL|
|           HV0005|              B02510|2021-01-03 08:53:34|2021-01-03 08:57:16|         191|         191|   NULL|
|           HV0003|              B02765|2021-01-06 16:10:19|2021-01-06 16:17:58|

In [43]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [44]:
crazy_stuff('B02884')

's/b44'

In [45]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [48]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

Py4JJavaError: An error occurred while calling o288.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 8.0 failed 1 times, most recent failure: Lost task 0.0 in stage 8.0 (TID 8) (host.docker.internal executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:685)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$2.read(PythonUDFRunner.scala:128)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$2.read(PythonUDFRunner.scala:106)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:596)
	at org.apache.spark.InterruptibleIterator.hasN

In [31]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003')

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-06 10:38:29|2021-01-06 10:45:44|         259|         259|
|2021-01-05 20:42:51|2021-01-05 20:58:14|          75|         147|
|2021-01-04 16:21:23|2021-01-04 16:29:28|          61|          72|
|2021-01-06 16:10:19|2021-01-06 16:17:58|         163|          90|
|2021-01-03 04:03:08|2021-01-03 04:32:51|         142|         132|
|2021-01-04 07:44:05|2021-01-04 08:14:36|          35|          65|
|2021-01-05 17:49:35|2021-01-05 17:58:27|          51|         259|
|2021-01-05 16:28:38|2021-01-05 16:42:39|          37|         225|
|2021-01-05 11:25:07|2021-01-05 11:42:44|         164|         211|
|2021-01-03 03:01:37|2021-01-03 03:16:48|          75|          48|
|2021-01-05 21:10:17|2021-01-05 21:20:46|         231|          79|
|2021-01-06 10:57:07|2021-01-06 11:08:04|       

In [30]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,
